# **Imports**

In [2]:
# Imports
from pathlib import Path
import json
import re
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from jiwer import cer, wer


In [28]:
# Imports for model
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn as nn
from tokenizer import TokenizerBPE01
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle as pkl
import json

# CTC
from utils import *

# Models
from models import models as md

from datasets import load_dataset

# **Models comparison**

In [35]:
ROOT_DIR = Path(os.getcwd())

VOCAB_DIRS = {
    "vocab (70 tokens)": ROOT_DIR / "trained_models/vocab",
    #"vocab1 (765 tokens)": ROOT_DIR / "trained_models/vocab1",
}

MODEL_INFO_JSON = ROOT_DIR / "models_config.json"

champions = {
    "vocab (70 tokens)": 13,  # Selected from plot below
}

TOP_N = 2

In [15]:
# Get models info
def load_model_info(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def format_model_name(model_idx, model_info):
    info = model_info[str(model_idx)]

    model_type = info["model_type"]
    model_no = info["model_no"]
    params = info.get("model_parameters", {})

    if params:
        param_string = ", ".join(
            f"{key}={value}"
            for key, value in params.items()
        )
        return f"{model_no}: {model_type} ({param_string})"

    return f"{model_no}: {model_type}"


# get results files
def find_result_files(vocab_dir):
    results = []

    pattern = re.compile(r"results_model(\d+)\.csv$", re.IGNORECASE)

    for csv_path in vocab_dir.rglob("results_model*.csv"):
        match = pattern.match(csv_path.name)
        if match is None:
            continue

        model_idx = int(match.group(1))

        results.append({
            "model_idx": model_idx,
            "path": csv_path
        })

    return sorted(results, key=lambda x: x["model_idx"])


# Get full model result info
def full_model_info(vocab_dir, model_info):
    models = {}

    files = find_result_files(vocab_dir)

    for item in files:
        model_idx = item["model_idx"]
        df = pd.read_csv(item["path"])

        models[model_idx] = {
            "name": format_model_name(model_idx, model_info),
            "path": item["path"],
            "data": df
        }
    return models

In [16]:
# Plotting graphs
def plot_vocab_models(vocab, models, initial_visible):
    fig = go.Figure()

    for model_idx, model in models.items():
        visible = model_idx in initial_visible
        df = model["data"]
        name = model["name"]

        # Train
        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Train"],
                mode="lines+markers",
                name=f"{name} - Train",
                legendgroup=str(model_idx),
                hovertemplate=(
                    f"{name}<br>"
                    "Epoch: %{x}<br>"
                    "Train: %{y}<extra></extra>"
                ),
                visible=visible
                
            )
        )


       # Validation
        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Validation"],
                mode="lines+markers",
                name=f"{name} - Validation",
                legendgroup=str(model_idx),
                hovertemplate=(
                    f"{name}<br>"
                    "Epoch: %{x}<br>"
                    "Train: %{y}<extra></extra>"
                ),
                visible=visible
            )
        )

    fig.update_layout(
        title=f"Models comparison - {vocab}",
        xaxis_title="Epoch",
        yaxis_title="Score",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Model",

        width=1200,
        height=800
    )

    fig.show()


def plot_champion_models(champion_models):
    fig = go.Figure()

    for vocab_name, model_idx, model in champion_models:
        df = model["data"]

        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Train"],
                mode="lines+markers",
                name=f"{vocab_name} - Train"
            )
        )

        fig.add_trace(
            go.Scatter(
                x=df["Epoch"],
                y=df["Validation"],
                mode="lines+markers",
                name=f"{vocab_name} - Validation"
            )
        )


    fig.update_layout(
        title="Final champion models comparison",
        xaxis_title="Epoch",
        yaxis_title="Score",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Model",
        width=1200,
        height=800
    )

    fig.show()
    


In [17]:
model_info = load_model_info(MODEL_INFO_JSON)
all_vocab_models = {}

for vocab_name, vocab_dir in VOCAB_DIRS.items():
    print(f"[{vocab_name}]")

    if not vocab_dir.exists():
        print(f"Directory {vocab_dir} does not exist.")
        continue

    models = full_model_info(vocab_dir, model_info)
    all_vocab_models[vocab_name] = models

    # Get final validation score for each model
    final_scores = []

    for model_idx, model in models.items():
        df = model["data"]

        final_validation = df["Validation"].iloc[-1]

        final_scores.append({
            "model_idx": model_idx,
            "score": final_validation
        })

    # Lowest validation loss first
    final_scores.sort(key=lambda x: x["score"])

    # Keep only the top N visible initially
    initial_visible = {
        x["model_idx"]
        for x in final_scores[:TOP_N]
    }

    print(f"Found {len(models)} models")


    if models:
        plot_vocab_models(vocab_name, models, initial_visible)

    print()


[vocab (70 tokens)]
Found 30 models


Model  2: slowly decellerating learning curve slope for both train and validation.
Model 13: steep learning at first epoch => learning pattern faster with both train and validation curve down slowly afterward, choose model 13.

In [20]:
champion_models = []

for vocab_name, model_idx in champions.items():

    if model_idx is None:
        print(f"No champion model selected for {vocab_name}")
        continue

    models = all_vocab_models.get(vocab_name, {})

    champion_models.append((vocab_name, model_idx, models[model_idx]))

    if len(champion_models) == 2:
        plot_champion_models(champion_models)

*For comparison between two vocab size set of models.*  
Eventhough vocab size 70 model show lower ctc_loss, ctc loss is affected by vocab size - lower chance to select the correct token per step, and tokenized sequence length reduced by using tokens represent 2 or more characters. ctc loss is not reasonable for cross-vocabulary comparison.

# **Champion model selection**

In [38]:
champions_model_1 = ROOT_DIR / "trained_models/vocab/encoder_decoder_lstm/model13.pt"
#champions_model_2 = ROOT_DIR / "trained_models/vocab1/bilstm_projection/model25.pt"

TOK_PATH = ROOT_DIR / "vocab.json"
# vocab1_path = ROOT_DIR / "vocab1.json"

results = {
    "vocab 70 tokens - model13": {
        "wer": [],
        "cer": []
    },

    # "vocab 765 tokens - model25": {
    #         "wer": [],
    #         "cer": []
    #     }
}



In [29]:
# Loading validation data
# Load data
print("Loading data...")
train_set = load_dataset(
    "pollen-robotics/speech-commands-v0.02",
    split="train"
)

val_set = load_dataset(
    "pollen-robotics/speech-commands-v0.02",
    split="validation"
)
print()
# Make train dataset
train_set = train_set.filter(
    lambda x: len(x["audio"]["array"]) <= 32000
)

val_set = val_set.filter(
    lambda x: len(x["audio"]["array"]) <= 32000
)

# Device
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print("Device:", device)
print()

Loading data...

Device: cpu



In [30]:
special_tokens = ['<PAD>', '<BOS>', '<EOS>']
if TOK_PATH:
    print("Getting tokenizer...")
    tokenizer = TokenizerBPE01()
    tokenizer.loadFromJSON(TOK_PATH)
else:
    print("Initiating tokenizer...")
    tokenizer = TokenizerBPE01()
    TOK_PATH = "vocab.json"
    tokenizer.saveToJSON(TOK_PATH)

tokenizer.update(special_tokens)
pad_id = tokenizer.vocab["<PAD>"]
bos_id = tokenizer.vocab["<BOS>"]
eos_id = tokenizer.vocab["<EOS>"]
print()
print("Tokenizer information")
print("----------------------------------------")
print("Vocabulary size:", len(tokenizer.vocab))
print("PAD ID         :", pad_id)
print("BOS ID         :", bos_id)
print("EOS ID         :", eos_id)
print("----------------------------------------")
print()

Getting tokenizer...
Initiate vocabulary of length: 38
Loaded.
Added 3/3 into vocabulary.
Existed: 0/3.
Current vocabulary length: 41

Tokenizer information
----------------------------------------
Vocabulary size: 41
PAD ID         : 38
BOS ID         : 39
EOS ID         : 40
----------------------------------------



In [31]:
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=16000,
    n_mfcc=13,
    melkwargs={
    "n_fft": 512,
    "win_length": 400,
    "hop_length": 160,
    "n_mels": 40
}
    )

In [32]:
# Data preparation
# Normalizer
class Normalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

    def to(self, device):
        self.mean = self.mean.to(device)
        self.std = self.std.to(device)

with open(ROOT_DIR / "normalizer1.pkl", "rb") as f:
    normalizer = pkl.load(f)
    normalizer.to(device) 


# Data preparation
def get_input_and_target(dataset):
    input_audios = dataset["audio"]
    target_sentences = dataset["label"]

    return input_audios, target_sentences


class SpeechDataset(Dataset):

    def __init__(
        self,
        input_audios,
        target_sentences,
        tokenizer,
        mfcc_transform,
        device,
        eos_id,
    ):

        self.input_audios = input_audios
        self.target_sentences = target_sentences

        self.tokenizer = tokenizer
        self.mfcc_transform = mfcc_transform

        self.device = device
        self.eos_id = eos_id

    def __len__(self):
        return len(self.input_audios)

    def __getitem__(self, index):

        try:

            input_audio = torch.tensor(
                self.input_audios[index]
            )

        except:
            input_audio = torch.tensor(
                self.input_audios[index]["array"]
            )
        mfccs = self.mfcc_transform(
            input_audio
        ).transpose(0, 1)


        token_ids = self.tokenizer.tokenize(
            self.target_sentences[index]
        )

        # Add EOS to the actual target
        token_ids = token_ids.tolist()
        token_ids.append(self.eos_id)
        token_ids = token_ids + [self.eos_id]

        tokens = torch.tensor(
            token_ids,
            dtype=torch.long,
        )

        return mfccs, tokens

def preprocess(audio, mfcc_transform, device):
        try:

            input_audio = torch.tensor(
                audio
            )

        except:
            input_audio = torch.tensor(
                audio["array"]
            )

        mfccs = mfcc_transform(input_audio).to(device).transpose(0, 1)

        seq_len = len(mfccs)

        return mfccs, seq_len

In [33]:
audios, targets = get_input_and_target(val_set)
del val_set

In [40]:
# Models loading
# Model 1
with open('models_config.json', 'r') as file:
    cfg = json.load(file)

model = cfg[str(13)]
model_no = model["model_no"]
model_type = model["model_type"]
model_parameters = model["model_parameters"]

model1 = md[model_type](
    input_size=13,
    output_size=len(tokenizer.vocab),
    **model_parameters
).to(device)

model1.load_state_dict(
    torch.load(champions_model_1, map_location=device)
)


# # Model 2
# model = cfg[str(25)]
# model_no = model["model_no"]
# model_type = model["model_type"]
# model_parameters = model["model_parameters"]

# model2 = models[model_type](
#     input_size=13,
#     output_size=len(tokenizer1.vocab),
#     **model_parameters
# ).to(device)

# model2.load_state_dict(
#     torch.load(champions_model_2, map_location=device)
# )

<All keys matched successfully>

In [46]:
for audio, target in zip(audios, targets):

    # Preprocess
    mfccs = preprocess(audio, mfcc_transform, device)[0]
    mfccs = mfccs.unsqueeze(0)

    # Inference
    prediction = greedy_decode(model1, mfccs, normalizer, bos_id, eos_id, max_length=50)
    prediction = prediction[0].tolist()

    if eos_id in prediction:

        prediction = prediction[
            :prediction.index(eos_id)
        ]


    prediction = tokenizer.tokenToWords(
            prediction
        )


    # Metrics
    wer1 = wer(target, prediction)
    cer1 = cer(target, prediction)

    # wer2 = wer(target, prediction1)
    # cer2 = cer(target, prediction1)

    # Save
    results["vocab 70 tokens - model13"]["wer"].append(wer1)
    results["vocab 70 tokens - model13"]["cer"].append(cer1)

    # results["vocab 765 tokens - model25"]["wer"].append(wer2)
    # results["vocab 765 tokens - model25"]["cer"].append(cer2)


In [47]:
def make_report(results):
    report = []

    for model_name, metrics in results.items():

        wer = np.array(metrics["wer"], dtype=float)
        cer = np.array(metrics["cer"], dtype=float)

        report.append({
            "Model": model_name,

            "Samples": len(wer),

            "WER Mean": wer.mean(),
            "WER Median": np.median(wer),
            "WER Std": wer.std(),
            "WER P25": np.percentile(wer, 25),
            "WER P75": np.percentile(wer, 75),
            "WER Min": wer.min(),
            "WER Max": wer.max(),

            "CER Mean": cer.mean(),
            "CER Median": np.median(cer),
            "CER Std": cer.std(),
            "CER P25": np.percentile(cer, 25),
            "CER P75": np.percentile(cer, 75),
            "CER Min": cer.min(),
            "CER Max": cer.max(),
        })

    return pd.DataFrame(report)


report = make_report(results)

print(report.to_string(index=False))

                    Model  Samples  WER Mean  WER Median  WER Std  WER P25  WER P75  WER Min  WER Max  CER Mean  CER Median  CER Std  CER P25  CER P75  CER Min  CER Max
vocab 70 tokens - model13     9981  0.181645         0.0 0.385552      0.0      0.0      0.0      1.0  0.183528         0.0 0.436146      0.0      0.0      0.0      4.0


Vocab 70 tokens is the better model.